In [6]:
from lib.db.crud.publication import get_or_create_publication
from lib.db.crud.authors.get_or_create import get_or_create_author
from datetime import date
import json
from lib.parser.lattes.livros import extrai_livros
from lib.parser.lattes.livros import parser_livros

from lib.db.crud.authors.vinculate_publication import authors_to_publication
from lib.db.crud.container import get_or_create_container
from lib.parser.crossref.author_crossref import parser_contributor
from lib.parser.lattes.container import parser_container_lattes
from lib.parser.lattes.contributor import parser_contributor_lattes
from lib.parser.lattes.publi_sem_doi import parser_publi_sem_doi

In [7]:
id_lattes = '2747150211073176'

In [ ]:
from bs4 import BeautifulSoup


def read_cv(path_cv):
    with open(path_cv, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
    return soup

path_html = 'data/curriculos/2747150211073176/cv.html'
soup = read_cv(path_html)

In [1]:
from sqlalchemy.orm import sessionmaker
from lib.db.database import engine

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)
session = SessionLocal()

# Create Profile

In [ ]:
from lib.parser.lattes.profile import save_profile

In [ ]:
save_profile(soup, lattes_id)

In [ ]:
from lib.db.crud.authors.profile import get_or_create_profile

author_db = get_or_create_profile(session, lattes_id)
author_db

# Atualiza Curiculos

In [ ]:
from lib.db.make_session import local_session
from lib.updates.check_cv import check_cv_update

In [ ]:
session = local_session()

html, update = check_cv_update(session, lattes_id)

# Artigos sem doi

In [ ]:
from lib.parser.lattes.artigos_completos import get_artigos_completos, slipt_artigos
list_artigos = get_artigos_completos(soup)
c_doi, s_doi = slipt_artigos(list_artigos)

In [ ]:
import json


with open("data/curriculos/2747150211073176/article_sem_doi.jsonl", "w", encoding="utf-8") as f:
    for article in s_doi:
        del article['raw_artigo']
        json.dump(article, f, ensure_ascii=False)
        f.write("\n")
        
    

In [ ]:
for article in s_doi:
    publication = parser_publi_sem_doi(article)
    container = parser_container_lattes(article)
    
    publication_db = get_or_create_publication(session, publication)
    container_db = get_or_create_container(session, container)
    publication_db.container = container_db
    
    authors = article["autores"]
    for author in authors:
        parsed_author = parser_contributor_lattes(author)
        contributor = parser_contributor(author, parsed_author)
        author_db, created = get_or_create_author(session, parsed_author)
        link = authors_to_publication(session, publication_db, author_db, contributor )
        print("FEITO: ", link)
        
    session.add(publication_db)
    session.commit() 
    
    


# Artigos com doi

In [ ]:
from lib.crossref.get import get_article_crossref


error = get_article_crossref(c_doi, lattes_id)

In [ ]:
import json


articles = []
with open('data/curriculos/2747150211073176/article_crossref.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        artigo = json.loads(line)
        articles.append(artigo)
len(articles)

In [ ]:
from lib.db.crud.articles import injest_article


injest_article(articles)

In [ ]:
error = []
with open("data/artigos/val.jsonl", "w", encoding="utf-8") as f:
    
    for i in c_doi:
        doi = i['doi'][0]
        url = f"https://api.crossref.org/v1/works/{doi}"
        r = httpx.get(url)
        print(r.status_code)
        if r.status_code == 200:
            item = r.json()['message']
            json.dump(item, f)
            f.write("\n")
        else:
            print(f"Error fetching data for DOI: {doi}, status code: {r.status_code}")
            error.append(i)

# Livros

In [ ]:
livros = extrai_livros(id_lattes)
livros = parser_livros(livros)
len(livros)

In [ ]:
with open(f'data/curriculos/{id_lattes}/livros.jsonl', 'w', encoding='utf-8') as f:
    for livro in livros:
        json.dump(livro, f, ensure_ascii=False)
        f.write('\n')
        

In [ ]:
livros = []
with open(f'data/curriculos/{id_lattes}/livros.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        livro = json.loads(line)
        livros.append(livro)
        
len(livros)    

In [ ]:
for livro in livros:
    print('INJGEST:', livros.index(livro))
    number_of_pages = livro.get('paginas') 
    if number_of_pages == '':
        number_of_pages = None
    publication = {
        'publication_type': 'book',
        'title': livro.get('titulo'),
        'date_published': date(int(livro.get('ano')), 1, 1),
        'doi': livro.get('doi'),
        'publisher': livro.get('editora'),
        'volume_number': livro.get('volume'),
        'number_of_pages': number_of_pages,
        'edition': livro.get('edicao'),
        'source': 'lattes'        
    }
    
    publication_db, created = get_or_create_publication(session, publication)
    if created:
        authors = livro["autores"]
        for author in authors:
            author_db, created = get_or_create_author(session, author)
            contributor = {'role': 'author'}
            link = authors_to_publication(session, publication_db, author_db, contributor )

# Capítulos de livros

In [8]:
capitulos = []
with open(f'data/curriculos/{id_lattes}/capitulos.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        capitulos.append(json.loads(line))
        
len(capitulos)  

70

In [ ]:
from lib.parser.lattes.capitulo_livros import normalize_capitulo


for capitulo in capitulos:
    publication = normalize_capitulo(capitulo)
    container = {'name': capitulo.get('titulo-do-livro')}
    publication_db, created = get_or_create_publication(session, publication)
    if created:
        container_db = get_or_create_container(session, container)
        publication_db.container = container_db
        authors = capitulo["autores"]
        for author in authors:
            author_db, created = get_or_create_author(session, author)
            contributor = {'role': 'author'}
            link = authors_to_publication(session, publication_db, author_db, contributor )
        session.add(publication_db)
        session.commit() 
            
    print("INJEST:", capitulos.index(capitulo))

In [2]:
from lib.db.models import Publication
from sqlalchemy import select, func

In [10]:

stm = (
    select(
        Publication.publication_type,
        func.count(Publication.id).label("total")
    )
    .group_by(Publication.publication_type)
    .order_by(func.count(Publication.id).desc())
)

resultados = session.execute(stm).all()

2026-04-29 16:16:44,194 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-29 16:16:44,198 INFO sqlalchemy.engine.Engine SELECT publications.publication_type, count(publications.id) AS total 
FROM publications GROUP BY publications.publication_type ORDER BY count(publications.id) DESC
2026-04-29 16:16:44,199 INFO sqlalchemy.engine.Engine [cached since 380.1s ago] {}


In [11]:
resultados

[('journal-article', 291), ('chapter-book', 69), ('book', 22)]

# Textos em jornais de notícias/revistas

In [ ]:
news = []
with open(f'data/curriculos/{id_lattes}/text_news.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        news.append(json.loads(line))
        
len(news)  